# Interventions & Potential Outcomes

Companion notebook for the [Interventions & Potential Outcomes lesson](https://ml-viz-ruby.vercel.app/courses/causal-inference/02-interventions-and-potential-outcomes).

**The idea in one sentence.** Every unit has two **potential outcomes** — $Y_1$ if
treated, $Y_0$ if not — but we only ever observe *one* of them (the "fundamental
problem of causal inference"), so estimating the average treatment effect
$\text{ATE} = \mathbb{E}[Y_1 - Y_0]$ requires either **adjustment** or
**randomization**.

Because this is a *simulation*, we know both potential outcomes and the true ATE —
so we can watch a confounded assignment bias the naive estimate, watch **backdoor
adjustment** recover the truth, and watch **randomization** ($do(T)$) make
adjustment unnecessary.

We build it from scratch, **validate that adjustment and randomization both
recover the true ATE**, then cover the gotchas. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## 1 — A simulation with known potential outcomes

A confounder Z affects treatment and outcome. The treatment has a TRUE effect of +0.2. We generate
both Y(0) and Y(1) for every unit (only possible in simulation) so we know the ground-truth ATE.

In [ ]:
n = 20000
Z = rng.random(n)                                   # confounder (e.g. severity)
TRUE_EFFECT = 0.2
Y0 = 0.3 + 0.5 * Z + rng.normal(0, 0.05, n)         # outcome if untreated
Y1 = Y0 + TRUE_EFFECT                                # outcome if treated (constant effect)
ate_true = (Y1 - Y0).mean()
print(f'TRUE ATE (we can compute it because we simulated both potential outcomes): {ate_true:.3f}')

## 2 — Confounded assignment → biased naive estimate

Higher-Z units are treated more often. We only get to observe the potential outcome matching each
unit's actual treatment — and the naive difference is biased upward by the confounding.

In [ ]:
T = (rng.random(n) < Z).astype(int)                 # confounded: treatment depends on Z
Y = np.where(T == 1, Y1, Y0)                        # observe only the realized outcome

naive = Y[T==1].mean() - Y[T==0].mean()
print(f'naive estimate: {naive:.3f}  (biased: true is {ate_true:.3f})')
print(f'confounding bias: {naive - ate_true:+.3f}')

## 3 — Backdoor adjustment recovers the ATE

Estimate the effect within strata of the confounder Z, then average over Z's distribution — the
backdoor formula. Since Z is the only confounder, this recovers the true effect.

In [ ]:
def backdoor_adjust(Z, T, Y, bins=20):
    edges = np.linspace(0, 1, bins + 1)
    effs, wts = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (Z >= lo) & (Z < hi)
        if (T[m]==1).sum() and (T[m]==0).sum():
            effs.append(Y[m & (T==1)].mean() - Y[m & (T==0)].mean())
            wts.append(m.sum())
    return np.average(effs, weights=wts)

print(f'backdoor-adjusted estimate: {backdoor_adjust(Z, T, Y):.3f}  (recovers true {ate_true:.3f})')

### Validate: backdoor adjustment recovers the true ATE

We know the truth ($\text{ATE}=0.2$). We assert the naive estimate is biased,
backdoor adjustment recovers the true ATE, and a **regression** of $Y$ on $T$ and
$Z$ recovers it too — three ways to the same number when the confounder is
measured.

In [ ]:
from sklearn.linear_model import LinearRegression

adj = backdoor_adjust(Z, T, Y)
print(f'true ATE:                {ate_true:.3f}')
print(f'naive estimate:          {naive:.3f}  (biased by confounding)')
print(f'backdoor-adjusted:       {adj:.3f}  (recovers the truth)')
assert abs(naive - ate_true) > 0.05, 'confounding should bias the naive estimate'
assert abs(adj - ate_true) < 0.03, 'backdoor adjustment should recover the true ATE'

coef_T = LinearRegression().fit(np.column_stack([T, Z]), Y).coef_[0]
print(f'regression coeff on T (control for Z): {coef_T:.3f}')
assert abs(coef_T - ate_true) < 0.03, 'regression adjustment should also recover the ATE'
print('\n✅ backdoor adjustment and regression both recover the true ATE of 0.2')

## 4 — Randomization makes adjustment unnecessary

If instead we *randomize* treatment (ignoring Z), the groups are comparable by construction and the
naive estimate is already unbiased — this is why RCTs and A/B tests are the gold standard.

In [ ]:
T_rand = (rng.random(n) < 0.5).astype(int)          # do(T): assignment independent of Z
Y_rand = np.where(T_rand == 1, Y1, Y0)
print(f'naive estimate under randomization: {Y_rand[T_rand==1].mean() - Y_rand[T_rand==0].mean():.3f}')
print(f'(already unbiased — true {ate_true:.3f} — no adjustment needed)')

### Validate: randomization is unbiased *without* any adjustment

Under $do(T)$ (random assignment), $T$ is independent of $Z$, so the naive
difference is already an unbiased ATE estimate — no backdoor adjustment needed.
That's the entire reason RCTs are the gold standard.

In [ ]:
naive_rand = Y_rand[T_rand == 1].mean() - Y_rand[T_rand == 0].mean()
print(f'naive estimate under randomization: {naive_rand:.3f}  (true {ate_true:.3f})')
assert abs(naive_rand - ate_true) < 0.02, 'randomization makes the naive estimate unbiased'
# and it no longer correlates with the confounder
corr_TZ = np.corrcoef(T_rand, Z)[0, 1]
print(f'correlation between random T and confounder Z: {corr_TZ:+.3f}  (~0 by design)')
assert abs(corr_TZ) < 0.05, 'randomization breaks the T–Z dependence'
print('\n✅ randomization severs T from Z, so no adjustment is needed — the RCT guarantee')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **unmeasured confounding** | adjustment recovers the ATE *only* if all confounders are observed (demo below) |
| **positivity / overlap** | every covariate stratum needs both treated and untreated units, or the effect is unidentified there |
| **SUTVA / no interference** | one unit's treatment mustn't affect another's outcome (breaks under network effects) |
| **extrapolation** | adjustment beyond the support of the data is a model assumption, not evidence |
| **effect heterogeneity** | the ATE is an average; subgroups can differ wildly (CATE) |

Demo: adjusting for only the *visible* part of a confounder leaves residual bias —
the untestable "no unmeasured confounding" assumption.

In [ ]:
# The catch: adjustment only works for the confounder you actually MEASURE. Here the
# real confounder U drives both treatment and outcome, but we only observe a noisy PROXY
# of it. Adjusting on the proxy leaves residual bias — the untestable "no unmeasured
# confounding" assumption behind every observational estimate.
U = rng.random(n)                                    # the TRUE (hidden) confounder
Y0_u = 0.3 + 0.5 * U + rng.normal(0, 0.05, n)
Y1_u = Y0_u + ate_true
T_u = (rng.random(n) < U).astype(int)                # treatment depends on the hidden U
Y_u = np.where(T_u == 1, Y1_u, Y0_u)
proxy = np.clip(U + rng.normal(0, 0.3, n), 0, 1)     # we can only measure a noisy proxy of U

adj_true  = backdoor_adjust(U,     T_u, Y_u)
adj_proxy = backdoor_adjust(proxy, T_u, Y_u)
print(f'true ATE:                               {ate_true:.3f}')
print(f'adjust on the TRUE confounder U:        {adj_true:.3f}  (recovers it)')
print(f'adjust on a NOISY PROXY of U:           {adj_proxy:.3f}  (residual bias remains)')
assert abs(adj_proxy - ate_true) > abs(adj_true - ate_true), 'imperfect adjustment leaves bias'
print('\nUnmeasured confounding is the Achilles heel of observational causal inference:')
print('you can only adjust for what you measured well, and no test can rule out a hidden cause.')

## ✏️ Your turn

**Exercise.** Implement `ate(Y1, Y0)` (the average treatment effect from both potential outcomes) and
`confounding_bias(naive, true_ate)` (how far the naive estimate is from the truth). Then you'll have
the full decomposition: naive = true ATE + bias.

In [ ]:
def ate(Y1, Y0):
    # TODO(you): average of the individual treatment effects Y1 - Y0
    return ...

def confounding_bias(naive, true_ate):
    # TODO(you): the gap between the naive estimate and the true ATE
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(ate(Y1, Y0), TRUE_EFFECT, atol=1e-9)
b = confounding_bias(naive, ate(Y1, Y0))
assert b > 0                                          # confounding inflated the naive estimate
assert np.isclose(naive, ate(Y1, Y0) + b)             # naive = true ATE + bias
# adjustment removes most of the bias
assert abs(backdoor_adjust(Z, T, Y) - ate(Y1, Y0)) < abs(b)
print(f'\u2713 ATE={ate(Y1,Y0):.3f}, naive bias={b:+.3f}, adjustment recovers the truth')

<details>
<summary>Solution</summary>

```python
def ate(Y1, Y0):
    return (Y1 - Y0).mean()

def confounding_bias(naive, true_ate):
    return naive - true_ate
```

In real data you never see both Y1 and Y0, so you can't compute the ATE directly — you estimate it
by randomization or by adjusting for confounders. The simulation lets us cheat and verify the
estimators actually recover the known truth.

</details>

## Key takeaways

- **Two potential outcomes, one observed.** The ATE $=\mathbb{E}[Y_1-Y_0]$ needs
  the counterfactual we never see — the fundamental problem of causal inference.
- **Confounding biases the naive estimate**; **backdoor adjustment** (or
  regression on the confounder) recovers the true ATE — we matched 0.2 three ways.
- **Randomization ($do(T)$) is the gold standard:** it severs $T$ from $Z$, so the
  naive difference is already unbiased and no adjustment is needed.
- **The fatal assumption is "no unmeasured confounding."** Adjust only for what you
  measure; a hidden common cause leaves residual bias that no data test can detect.